# iSCORS-Net — Colab Runner

**Workflow:**
1. Zip your local `iscors-net` folder (exclude `checkpoint/` and `data/` if large).
2. Upload the `.zip` using the **Upload** button below (Cell 2), or via the Files sidebar.
3. Run all cells top-to-bottom.
4. Download the `results_<VERSION>.zip` produced by the last cell.

---

In [ ]:
# ── Version tag: bump this when you change hyperparameters or architecture ──
VERSION = 'v2.0'
print(f'iSCORS-Net  {VERSION}')

In [ ]:
# ── Upload zip via Colab file-picker ────────────────────────────────────────
from google.colab import files
uploaded = files.upload()   # opens a file-picker dialog
print('Uploaded:', list(uploaded.keys()))

In [ ]:
import os, glob

# Unzip
!unzip -q *.zip -d .

# Auto-discover project root (folder that contains main.py)
found = glob.glob('**/main.py', recursive=True)
if found:
    project_root = os.path.dirname(os.path.abspath(found[0]))
    os.chdir(project_root)
    print(f'Project root: {project_root}')
else:
    raise RuntimeError('main.py not found — check your zip.')

print('Files:', os.listdir('.'))

In [ ]:
# ── Install dependencies ─────────────────────────────────────────────────────
!pip install -q -r requirements.txt scipy
print('Dependencies installed.')

In [ ]:
# ── Generate synthetic test video (if not already present) ───────────────────
import os
if not os.path.exists('./data/test_synthetic_cell.tif'):
    os.makedirs('./data', exist_ok=True)
    !python utils/generate_test_video.py
else:
    print('Test video already exists — skipping generation.')

In [ ]:
# ── Internal Learning Training ───────────────────────────────────────────────
# Key settings (mirrors train_internal.py):
#   TRAIN_RATIO = 0.02   (2% sparse pixels)
#   RANDOM_TAU  = True   (randomly sampled tau delays each step)
#   Loss mask   = center 3x3 only
#   FFT + Hann window autocorrelation for GT
!python train_internal.py

In [ ]:
# ── Visualise saved results ──────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob, os

result_files = sorted(glob.glob('./result/*.png'))
print(f'Result images ({len(result_files)}):', [os.path.basename(f) for f in result_files])

for path in result_files:
    img = mpimg.imread(path)
    plt.figure(figsize=(10, 4))
    plt.imshow(img)
    plt.axis('off')
    plt.title(os.path.basename(path))
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Package & download results ───────────────────────────────────────────────
import zipfile, os, glob
from google.colab import files

zip_name = f'results_{VERSION}.zip'

with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    # Result images and checkpoints tagged with VERSION
    for pattern in [f'./result/*{VERSION}*', f'./checkpoint/*{VERSION}*']:
        for path in glob.glob(pattern):
            zf.write(path, os.path.relpath(path, '.'))
    # Also include the full result folder for convenience
    for path in glob.glob('./result/*.png'):
        arcname = os.path.relpath(path, '.')
        if arcname not in zf.namelist():
            zf.write(path, arcname)

print(f'Created {zip_name}  ({os.path.getsize(zip_name)/1024:.1f} KB)')
files.download(zip_name)